In [2]:
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path

API_URL = "https://restcountries.com/v3.1/all?fields=cca3,timezones"

@task
def extract_timezones():
    response = requests.get(API_URL)
    response.raise_for_status()
    return response.json()

@task
def transform_timezones(data):
    df = pd.DataFrame(data)
    df_tz = df.explode("timezones")
    df_tz = df_tz.rename(columns={"cca3": "id_country", "timezones": "timezone"})
    return df_tz

@task
def load_timezones(df: pd.DataFrame):
    file_path = Path("../stage/country_timezones.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"Saved {len(df)} rows to {file_path}")

@flow(name="etl_timezones_flow")
def etl_timezones():
    data = extract_timezones()
    df = transform_timezones(data)
    load_timezones(df)

if __name__ == "__main__":
    etl_timezones()


11:39:10.629 | INFO    | Flow run 'messy-walrus' - Beginning flow run 'messy-walrus' for flow 'etl_timezones_flow'

11:39:11.716 | INFO    | Task run 'extract_timezones-81c' - Finished in state Completed()

11:39:11.945 | INFO    | Task run 'transform_timezones-3c6' - Finished in state Completed()

Saved 340 rows to ..\stage\country_timezones.csv


11:39:12.161 | INFO    | Task run 'load_timezones-f0c' - Finished in state Completed()

11:39:12.174 | INFO    | Flow run 'messy-walrus' - Finished in state Completed()